# AI形式化数学工具使用示例

本笔记本演示如何使用AI形式化数学工具链进行数学问题的自动形式化。

## 1. 环境设置

In [ ]:
import sys
import os
from pathlib import Path

# 添加tools目录到路径
sys.path.insert(0, str(Path.cwd().parent / "tools"))

# 导入工具
from deepseek_client import DeepSeekClient, FormalizationMode, formalize_problem
from lean_verifier import LeanVerifier, verify_lean_code, check_syntax
from imo_dataset import IMODatasetProcessor, load_imo_problems
from msc_recommender import MSCRecommender, recommend_msc
from auto_formalize import AutoFormalizationWorkflow, quick_formalize

print("✅ 所有工具导入成功！")

### 1.1 设置API密钥（如果需要）

In [ ]:
# 设置DeepSeek API密钥
# 方式1: 直接设置（不推荐用于生产环境）
# os.environ["DEEPSEEK_API_KEY"] = "your-api-key-here"

# 方式2: 从.env文件加载
from dotenv import load_dotenv
load_dotenv()

if os.getenv("DEEPSEEK_API_KEY"):
    print("✅ API密钥已设置")
else:
    print("⚠️  未设置DEEPSEEK_API_KEY环境变量")
    print("   部分功能将使用模拟模式")

## 2. DeepSeek API客户端使用

### 2.1 基本形式化

In [ ]:
# 初始化客户端
client = DeepSeekClient()

# 示例数学问题
problem = """
证明：对于任意正整数n，n³ - n 总是能被6整除。
"""

print("原始问题:")
print(problem)
print("\n" + "="*60 + "\n")

# 形式化转换
result = client.formalize(problem, mode=FormalizationMode.NATURAL_TO_LEAN)

if result.success:
    print("✅ 形式化成功！")
    print(f"耗时: {result.latency_ms:.2f}ms")
    print(f"Token使用: {result.tokens_used}")
    print("\n生成的Lean 4代码:")
    print("-"*60)
    print(result.content)
else:
    print(f"❌ 失败: {result.error_message}")

### 2.2 证明生成

In [ ]:
# 定理陈述
theorem = """
import Mathlib

theorem number_theory_1234 (n : ℕ) :
    6 ∣ (n^3 - n) := by
    sorry
"""

print("待证明的定理:")
print(theorem)

# 生成证明
proof_result = client.formalize(
    theorem, 
    mode=FormalizationMode.PROOF_GENERATION
)

if proof_result.success:
    print("\n✅ 证明生成成功！")
    print("-"*60)
    print(proof_result.content)
else:
    print(f"\n❌ 失败: {proof_result.error_message}")

### 2.3 错误修正

In [ ]:
# 有错误的代码
buggy_code = """
import Mathlib

theorem test (n : ℕ) : n + n = 2 * n := by
    ring  -- 这里可能出错如果没有导入正确的库
"""

error_msg = "unknown tactic 'ring'"

print("有错误的代码:")
print(buggy_code)
print(f"\n错误信息: {error_msg}")

# 尝试修正
fix_result = client.formalize(
    buggy_code,
    mode=FormalizationMode.ERROR_CORRECTION,
    error=error_msg
)

if fix_result.success:
    print("\n✅ 修正后的代码:")
    print("-"*60)
    print(fix_result.content)
else:
    print(f"\n❌ 修正失败: {fix_result.error_message}")

## 3. Lean4验证器使用

In [ ]:
# 初始化验证器
verifier = LeanVerifier()

# 示例Lean代码
lean_code = """
import Mathlib

theorem arithmetic_mean_ineq (a b : ℝ) (ha : 0 < a) (hb : 0 < b) :
    2 * a * b / (a + b) ≤ (a + b) / 2 := by
  have h : 0 < a + b := add_pos ha hb
  apply (div_le_div_iff (by positivity) (by positivity)).mpr
  linarith [sq_nonneg (a - b)]
"""

print("待验证的Lean代码:")
print("-"*60)
print(lean_code)

# 执行验证
result = verifier.verify(lean_code)

print("\n" + "="*60)
print(f"验证状态: {result.status.value}")
print(f"是否有效: {result.is_valid}")
print(f"耗时: {result.elapsed_time_ms:.2f}ms")
print(f"错误数: {len(result.errors)}")
print(f"警告数: {len(result.warnings)}")

In [ ]:
# 生成验证报告
if result.errors:
    print("\n错误详情:")
    for error in result.errors:
        print(f"  行{error.line}, 列{error.column}: {error.message}")

# 生成markdown格式的报告
report = verifier.generate_report(result, output_format="markdown")
print("\n" + "="*60)
print("验证报告:")
print("="*60)
print(report)

## 4. IMO数据集处理

In [ ]:
# 初始化数据集处理器
processor = IMODatasetProcessor()

# 加载所有IMO题目
problems = processor.load_problems()

print(f"✅ 加载了 {len(problems)} 道IMO题目")

# 显示第一个题目
if problems:
    print("\n" + "="*60)
    print("示例题目:")
    print("="*60)
    p = problems[0]
    print(f"ID: {p.unique_id}")
    print(f"标题: {p.title}")
    print(f"类别: {p.category}")
    print(f"概念: {', '.join(p.concepts[:5])}")
    print(f"\n题目内容:")
    print(p.statement[:300] + "..." if len(p.statement) > 300 else p.statement)

In [ ]:
# 按类别统计
stats = processor.generate_statistics()

print("\n题目统计:")
print("="*60)
print(f"总数: {stats['total_problems']}")
print("\n按类别分布:")
for category, count in stats['by_category'].items():
    print(f"  {category}: {count}")
print("\n按难度分布:")
for difficulty, count in stats['by_difficulty'].items():
    print(f"  {difficulty}: {count}")

In [ ]:
# 搜索题目
search_results = processor.search_problems("prime")
print(f"\n搜索 'prime' 找到 {len(search_results)} 道题目")

for p in search_results[:3]:
    print(f"\n- {p.unique_id}: {p.title}")
    print(f"  概念: {', '.join(p.keywords[:5])}")

## 5. MSC编码推荐

In [ ]:
# 初始化推荐器
recommender = MSCRecommender()

# 示例数学文本
math_texts = [
    "Prove that for any prime number p, the congruence x^2 ≡ -1 (mod p) has a solution if and only if p = 2 or p ≡ 1 (mod 4).",
    "Find all functions f: R -> R such that f(x+y) = f(x) + f(y) for all real numbers x, y.",
    "Let G be a finite group of order n. Prove that if p is a prime dividing n, then G has an element of order p.",
    "Find the number of ways to color the vertices of a regular hexagon with 3 colors such that adjacent vertices have different colors."
]

for i, text in enumerate(math_texts, 1):
    print(f"\n{'='*60}")
    print(f"文本 {i}:")
    print(text[:80] + "..." if len(text) > 80 else text)
    print("-"*60)
    
    recommendations = recommender.recommend(text, top_k=3)
    
    for rec in recommendations:
        print(f"\n  {rec.code}: {rec.description}")
        print(f"    置信度: {rec.confidence:.2f}")
        print(f"    匹配词: {', '.join(rec.matched_keywords[:3])}")

## 6. 自动化工作流

In [ ]:
# 初始化自动化工作流
from auto_formalize import WorkflowConfig

config = WorkflowConfig(
    output_dir="./output",
    verify_after_formalization=True,
    auto_fix_errors=True,
    generate_report=True
)

workflow = AutoFormalizationWorkflow(config)

# 定义回调函数
def on_task_start(task):
    print(f"🚀 开始处理: {task.task_id}")

def on_task_complete(task):
    status_icon = "✅" if task.status == "success" else "❌"
    print(f"{status_icon} 完成: {task.task_id} ({task.status})")

workflow.on_task_start = on_task_start
workflow.on_task_complete = on_task_complete

In [ ]:
# 处理单个问题
test_problem = """
证明：对于任意正整数n，如果n²是偶数，则n也是偶数。
"""

print("处理测试问题...")
task = workflow.process_problem(test_problem, source="User Test")

print("\n" + "="*60)
print("处理结果:")
print("="*60)
print(f"状态: {task.status}")
print(f"MSC推荐:")
for rec in task.msc_recommendations[:3]:
    print(f"  - {rec.code}: {rec.description} (置信度: {rec.confidence:.2f})")

if task.generated_lean_code:
    print("\n生成的Lean 4代码:")
    print("-"*60)
    print(task.generated_lean_code)

In [ ]:
# 批量处理示例
test_problems = [
    {
        "problem_text": "证明：对于任意正整数n，n^3 - n能被6整除。",
        "source": "Number Theory Example",
        "metadata": {"category": "number_theory"}
    },
    {
        "problem_text": "在三角形ABC中，证明：a^2 = b^2 + c^2 - 2bc*cos(A)。",
        "source": "Geometry Example",
        "metadata": {"category": "geometry"}
    },
    {
        "problem_text": "求1到100之间所有素数的和。",
        "source": "Combinatorics Example",
        "metadata": {"category": "combinatorics"}
    }
]

print("开始批量处理...")
result = workflow.batch_process(test_problems)

print("\n" + "="*60)
print("批量处理结果:")
print("="*60)
print(f"总任务数: {result.total_tasks}")
print(f"成功: {result.successful_tasks}")
print(f"失败: {result.failed_tasks}")
print(f"总耗时: {result.total_time_ms / 1000:.2f}秒")
print(f"报告路径: {result.report_path}")

## 7. 快速形式化函数

In [ ]:
# 使用快速形式化函数
problem = "证明勾股定理：在直角三角形中，斜边的平方等于两直角边的平方和。"

print(f"问题: {problem}")
print("\n处理中...\n")

result = quick_formalize(problem, verify=True)

print("="*60)
if result["success"]:
    print("✅ 形式化成功！")
    print(f"\nMSC编码:")
    for msc in result["msc_codes"]:
        print(f"  - {msc['code']}: {msc['description']}")
    
    if result["verification"]:
        print(f"\n验证状态: {result['verification']['status']}")
    
    print("\n生成的代码:")
    print("-"*60)
    print(result["lean_code"])
else:
    print(f"❌ 失败: {result['error']}")

## 8. 实际IMO题目形式化

In [ ]:
# 加载一道IMO题目并进行形式化
if problems:
    # 选择一道数论题目
    nt_problems = [p for p in problems if p.category == "number_theory"]
    
    if nt_problems:
        selected = nt_problems[0]
        print(f"选择题目: {selected.unique_id}")
        print(f"类别: {selected.category}")
        print(f"概念: {', '.join(selected.concepts)}")
        print("\n题目内容:")
        print("="*60)
        print(selected.statement)
        
        print("\n" + "="*60)
        print("开始形式化...")
        
        task = workflow.process_problem(
            selected.statement,
            source=selected.unique_id,
            metadata={
                "year": selected.year,
                "number": selected.problem_number,
                "category": selected.category
            }
        )
        
        print("\n" + "="*60)
        print("形式化结果:")
        print("="*60)
        print(f"状态: {task.status}")
        
        if task.generated_lean_code:
            print("\n生成的Lean 4代码:")
            print("-"*60)
            print(task.generated_lean_code)

## 9. 导出结果

In [ ]:
# 导出IMO数据集
import json

# 导出为JSON
json_output = processor.export_to_json("imo_problems.json")
print("✅ 导出IMO题目到JSON")

# 导出训练数据
training_output = processor.export_training_data("training_data.jsonl")
print(f"✅ 导出训练数据到 {training_output}")

# 保存统计信息
stats_path = Path("./output/statistics.json")
stats_path.parent.mkdir(parents=True, exist_ok=True)
with open(stats_path, 'w', encoding='utf-8') as f:
    json.dump(stats, f, indent=2, ensure_ascii=False)
print(f"✅ 保存统计信息到 {stats_path}")

## 10. 总结

In [ ]:
print("\n" + "="*60)
print("AI形式化数学工具使用示例 - 总结")
print("="*60)
print("""
本笔记本演示了以下功能:

1. DeepSeek API客户端
   - 自然语言到Lean4的形式化转换
   - 证明生成
   - 错误修正

2. Lean4验证器
   - 代码语法验证
   - 类型检查
   - 生成验证报告

3. IMO数据集处理器
   - 加载和解析IMO题目
   - 概念和关键词提取
   - 数据集导出

4. MSC编码推荐器
   - 基于文本的MSC编码推荐
   - 置信度评估

5. 自动化工作流
   - 端到端自动形式化
   - 批量处理
   - 报告生成

所有工具都支持模块化使用，可以根据需求组合使用。
""")
print("="*60)